In [1]:
%pip install pyautogui pillow keyboard

Note: you may need to restart the kernel to use updated packages.


In [21]:
import pyautogui
import time
import keyboard  # 用來監聽手動中斷按鍵

# ==========================================
# 1. 基本參數設定 & 全局優化
# ==========================================
basex = 1287  # 左上角第一顆寶石的中心點 X 座標
basey = 390  # 左上角第一顆寶石的中心點 Y 座標
gap = 40     # 每顆寶石之間的間隔像素距離

animation_delay = 0.37  # 寶石交換與消除的等待時間(秒)，可視遊戲速度調低以加快效率
pyautogui.PAUSE = 0.02  # 降低 pyautogui 預設的指令延遲，讓滑鼠動作更俐落

# 計算只截取遊戲盤面的區域 (left, top, width, height)
# 這樣截圖速度會比全螢幕快非常多！
board_region = (
    basex - gap // 2, 
    basey - gap // 2, 
    gap * 8, 
    gap * 8
)

# ==========================================
# 2. 輔助與座標/顏色獲取函式
# ==========================================
def get_gem_pos(r, c):
    """取得 (r, c) 寶石在實體螢幕上的 (x, y) 座標，用於滑鼠點擊"""
    x = basex + c * gap
    y = basey + r * gap
    return x, y

def get_gem_color(r, c, im):
    """取得 (r, c) 寶石的中心點顏色"""
    if r < 0 or r >= 8 or c < 0 or c >= 8:
        return None
    # 因為現在是「區域截圖」，影像的起始點變成了 (0,0)
    # 所以要轉換成相對於這張小截圖的座標
    img_x = int((c * gap) + gap // 2)
    img_y = int((r * gap) + gap // 2)
    return im.getpixel((img_x, img_y))

# ==========================================
# 3. 三消邏輯判斷核心 (維持原架構)
# ==========================================
def get_simulated_color(r, c, im, pos1, pos2):
    if not (0 <= r < 8 and 0 <= c < 8):
        return None
    if (r, c) == pos1:
        return get_gem_color(pos2[0], pos2[1], im)
    if (r, c) == pos2:
        return get_gem_color(pos1[0], pos1[1], im)
    return get_gem_color(r, c, im)

def is_match_after_swap(r, c, im, pos1, pos2):
    target_color = get_simulated_color(r, c, im, pos1, pos2)
    if target_color is None:
        return False
        
    v_count = 1
    for i in range(1, 3):
        if get_simulated_color(r - i, c, im, pos1, pos2) == target_color: v_count += 1
        else: break
    for i in range(1, 3):
        if get_simulated_color(r + i, c, im, pos1, pos2) == target_color: v_count += 1
        else: break
    if v_count >= 3: return True
        
    h_count = 1
    for i in range(1, 3):
        if get_simulated_color(r, c - i, im, pos1, pos2) == target_color: h_count += 1
        else: break
    for i in range(1, 3):
        if get_simulated_color(r, c + i, im, pos1, pos2) == target_color: h_count += 1
        else: break
    if h_count >= 3: return True
        
    return False

# ==========================================
# 4. 操作函式 (優化速度)
# ==========================================
def swap_gems(r1, c1, r2, c2):
    """快速點擊兩顆寶石交換"""
    x1, y1 = get_gem_pos(r1, c1)
    x2, y2 = get_gem_pos(r2, c2)
    pyautogui.click(x1, y1)
    time.sleep(0.02) # 極短暫延遲即可
    pyautogui.click(x2, y2)

# ==========================================
# 5. 主程式迴圈 (升級至 90 秒 + 手動中斷)
# ==========================================
if __name__ == "__main__":
    print("Bot 開始運行... (隨時可按下 'q' 鍵停止程式)")
    
    start_time = time.time()
    # 使用 region 進行局部截圖，速度極大化
    im = pyautogui.screenshot(region=board_region)
    
    while True:
        # 手動中斷機制
        if keyboard.is_pressed('q'):
            print("玩家手動中斷程式！")
            break

        # 時間延長至 90 秒 (1.5 分鐘)
        if time.time() - start_time > 90:
            print("已執行滿 1 分 30 秒，結束迴圈。")
            break
            
        action_taken = False
        
        # 演算法升級：只需檢查「向右」跟「向下」，就能涵蓋 4 個方向 (提升一倍效能)
        for r in range(8):
            for c in range(8):
                
                # 檢查向右交換 (等同檢查右邊那顆向左)
                if c < 7:
                    if is_match_after_swap(r, c, im, (r, c), (r, c+1)) or \
                       is_match_after_swap(r, c+1, im, (r, c), (r, c+1)):
                        swap_gems(r, c, r, c+1)
                        action_taken = True
                        break
                        
                # 檢查向下交換 (等同檢查下面那顆向上)
                if r < 7:
                    if is_match_after_swap(r, c, im, (r, c), (r+1, c)) or \
                       is_match_after_swap(r+1, c, im, (r, c), (r+1, c)):
                        swap_gems(r, c, r+1, c)
                        action_taken = True
                        break
            
            if action_taken:
                break
                
        if action_taken:
            # 有動作時，給予遊戲動畫時間，再進行局部截圖
            time.sleep(animation_delay)
            im = pyautogui.screenshot(region=board_region)
        else:
            # 沒步可走時的短暫休眠，避免死結
            time.sleep(0.2)
            im = pyautogui.screenshot(region=board_region)

Bot 開始運行... (隨時可按下 'q' 鍵停止程式)
玩家手動中斷程式！
